# 00 Profile

Thin notebook over the phase script(s) in `scripts/`. Cells are split at the `# ----` section markers. Run from the repo root (`jupyter lab`), or just run the scripts.

In [ ]:
import os, sys
os.chdir(os.path.dirname(os.getcwd())) if os.path.basename(os.getcwd()) == 'notebooks' else None
sys.path.insert(0, os.getcwd())
%matplotlib inline

## scripts/00_profile_raw.py

In [ ]:
"""Phase 0, step 1: profile the raw dunnhumby workbook. Prints everything, builds no models.

Reads the actual Excel file (every sheet), caches each sheet as CSV in data/raw/csv/
so later steps do not pay the 30 MB xlsx parse again.
"""
import sys
from pathlib import Path

import pandas as pd

ROOT = Path(os.getcwd())
RAW = ROOT / "data" / "raw" / "extracted" / "dunnhumby_Breakfast-at-the-Frat"
CSV_DIR = ROOT / "data" / "raw" / "csv"
CSV_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

xlsx = next(RAW.glob("*.xlsx"))
print("File:", xlsx.name, f"({xlsx.stat().st_size/1e6:.1f} MB)")

sheets = pd.read_excel(xlsx, sheet_name=None)
for name, df in sheets.items():
    safe = name.replace(" ", "_").replace("-", "_").lower()
    df.to_csv(CSV_DIR / f"{safe}.csv", index=False)
    print("\n" + "=" * 80)
    print(f"SHEET: {name!r}  shape={df.shape}")
    print("-" * 80)
    print(df.dtypes.to_string())
    print("\nNull counts:")
    print(df.isna().sum().to_string())
    print("\nHead:")
    print(df.head(5).to_string())
    num = df.select_dtypes("number")
    if not num.empty:
        print("\nDescribe:")
        print(num.describe().T.to_string())
    for c in df.columns:
        if df[c].nunique() <= 30:
            print(f"\nvalue_counts[{c}]:")
            print(df[c].value_counts(dropna=False).to_string())
print("\nCSV cache written to", CSV_DIR)

## scripts/01_phase0_questions.py

In [ ]:
"""Phase 0, step 2: answer the structural questions in writing before any model is built.

Writes outputs/phase0_answers.md and figures to outputs/figures/.
"""
import sys
from pathlib import Path

import matplotlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path(os.getcwd())
sys.path.insert(0, str(ROOT))
from src.load import build_clean  # noqa: E402

FIG = ROOT / "outputs" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
OUT = ROOT / "outputs" / "phase0_answers.md"

df = build_clean()
lines = []
P = lines.append

P("# Phase 0 answers (generated by scripts/01_phase0_questions.py)\n")
P(f"- Rows (store x product x week): **{len(df):,}**")
P(f"- Stores: **{df.store_num.nunique()}**, products (UPCs): **{df.upc.nunique()}**, "
  f"categories: **{df.category.nunique()}** ({', '.join(sorted(df.category.unique()))})")
P(f"- Weeks: **{df.week_index.nunique()}** (week_index {df.week_index.min()}..{df.week_index.max()}), "
  f"{df.week_end_date.min().date()} to {df.week_end_date.max().date()}")
full = df.store_num.nunique() * df.upc.nunique() * df.week_index.nunique()
P(f"- Grid completeness: {len(df)/full:.1%} of the full store x product x week grid is observed "
  f"(missing cells = product not sold/stocked that week; treated as absent, not zero)")
P(f"- Price nulls repaired: price {int(df.price_imputed.sum())}, base_price {int(df.base_price_imputed.sum())}")

# --- promo share
share = df.is_promo.mean()
P(f"\n## 1. What fraction of store-weeks have a promotion?\n")
P(f"- Overall: **{share:.1%}** of store-product-weeks are promoted (any of feature / display / price cut >5% / TPR flag).")
by_cat = df.groupby("category").is_promo.mean().sort_values()
P("- By category:\n")
P(by_cat.to_frame("promo_share").to_markdown(floatfmt=".1%"))
P("\n- Promo type mix:\n")
P(df.promo_type.value_counts(normalize=True).to_frame("share").to_markdown(floatfmt=".1%"))
P(f"\n- Under 40%, so clean non-promo baseline weeks exist for every category. "
  f"{by_cat.index[-1].title()} is the most promoted ({by_cat.iloc[-1]:.0%}); {by_cat.index[0].title()} the least ({by_cat.iloc[0]:.0%}).")

# --- run lengths
runs = df[df.is_promo].groupby("promo_run_id").promo_run_length.first()
P("\n## 1b. How long do promotions run?\n")
P(f"- {runs.shape[0]:,} promotion runs. Run length distribution (weeks):\n")
P(runs.value_counts().sort_index().head(10).to_frame("runs").to_markdown())
P(f"\n- Share of runs lasting exactly one week: **{(runs==1).mean():.1%}**; mean run length {runs.mean():.2f} weeks.")
P("- Consequence: an *event* is defined as a promotion run (consecutive promoted weeks). In-week lift is "
  "measured over the run's weeks; the post-window starts the week after the run ends.")

# --- chain-wide vs store-specific
P("\n## 2. Are promotions chain-wide or store-specific? (the critical one)\n")
wk = df.groupby(["upc", "week_index"]).agg(n_stores=("store_num", "size"), n_promo=("is_promo", "sum"))
wk["share_promo"] = wk.n_promo / wk.n_stores
promo_wk = wk[wk.n_promo > 0]
P(f"- Of {len(wk):,} product-weeks, {len(promo_wk):,} have at least one promoting store.")
P(f"- In those product-weeks, the median share of stores promoting is **{promo_wk.share_promo.median():.0%}**, "
  f"mean {promo_wk.share_promo.mean():.0%}.")
bins = pd.cut(promo_wk.share_promo, [0, .1, .25, .5, .75, .9, 1.0], include_lowest=True)
P("- Distribution of 'share of stores promoting' across promoted product-weeks:\n")
P(bins.value_counts().sort_index().to_frame("product_weeks").to_markdown())
# event-level view: for each promoted store-product-week, how many OTHER stores were clean controls
ev = df[df.is_promo].merge(wk.reset_index()[["upc", "week_index", "n_stores", "n_promo"]], on=["upc", "week_index"])
ev["n_control"] = ev.n_stores - ev.n_promo
P(f"\n- From the promoted store-week's point of view: median **{ev.n_control.median():.0f}** non-promoting stores "
  f"sell the same product that week; {(ev.n_control >= 5).mean():.0%} of promoted store-weeks have at least 5 candidate control stores "
  f"and {(ev.n_control == 0).mean():.1%} have none.")
P("- Verdict: promotions are mostly **chain-wide-ish but not universal**. Enough events keep a usable control group "
  "for Method B, but controls must be screened for being non-promo in the pre-window too (done in Phase 1).")

# by promo type: are displays store-specific and features chain-wide?
tmp = df[df.is_promo].copy()
for flag in ["feature", "display", "price_cut"]:
    w = df.groupby(["upc", "week_index"])[flag].mean()
    tmp[f"share_{flag}"] = tmp.set_index(["upc", "week_index"]).index.map(w)
P("\n- Share of stores carrying the same lever in the same product-week, for promoted store-weeks that carry it:\n")
rows = []
for flag in ["feature", "display", "price_cut"]:
    s = tmp.loc[tmp[flag].astype(bool), f"share_{flag}"]
    rows.append({"lever": flag, "median_share_of_stores": s.median(), "mean": s.mean()})
P(pd.DataFrame(rows).to_markdown(index=False, floatfmt=".0%"))
P("\n- Features (circular ads) are near chain-wide; displays are the most store-specific lever. "
  "Display-only events are therefore the cleanest DiD events.")

# --- seasonality of promos
P("\n## 3. When do promotions cluster?\n")
woy = df.groupby("week_of_year").is_promo.mean()
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(woy.index, woy.values * 100, color="#1f6f8b")
ax.set_xlabel("ISO week of year"); ax.set_ylabel("% of store-product-weeks promoted")
ax.set_title("Promotion frequency by week of year (all 3 years pooled)")
ax.grid(alpha=.3); fig.tight_layout(); fig.savefig(FIG / "phase0_promo_by_week_of_year.png", dpi=130); plt.close(fig)
P(f"- Range across the year: {woy.min():.1%} to {woy.max():.1%}. Peak weeks: "
  f"{', '.join(str(i) for i in woy.sort_values(ascending=False).head(5).index)}; "
  f"troughs: {', '.join(str(i) for i in woy.sort_values().head(5).index)}.")
# demand seasonality for context
dem = df[~df.is_promo].groupby("week_of_year").units.mean()
corr = np.corrcoef(woy.values, dem.reindex(woy.index).values)[0, 1]
P(f"- Correlation between promo frequency and non-promo mean units across weeks of year: **{corr:+.2f}**. "
  "This is the quantitative check on 'retailers promote in high-demand weeks'. A positive value means the "
  "baseline must control week-of-year or lift will be overstated.")
P("- Figure: outputs/figures/phase0_promo_by_week_of_year.png")

cat_woy = df.groupby(["category", "week_of_year"]).is_promo.mean().unstack(0)
fig, ax = plt.subplots(figsize=(10, 3.5))
cat_woy.mul(100).plot(ax=ax)
ax.set_xlabel("ISO week of year"); ax.set_ylabel("% promoted"); ax.set_title("Promotion frequency by week of year, by category")
ax.grid(alpha=.3); fig.tight_layout(); fig.savefig(FIG / "phase0_promo_by_woy_category.png", dpi=130); plt.close(fig)

# --- base price stability
P("\n## 4. Is base price stable per product-store?\n")
bp = df.groupby(["store_num", "upc"]).base_price.agg(["nunique", "min", "max", "std", "mean"])
bp["cv"] = bp["std"] / bp["mean"]
P(f"- Median number of distinct base prices per store-product over 156 weeks: **{bp['nunique'].median():.0f}** "
  f"(mean {bp['nunique'].mean():.1f}); median coefficient of variation {bp.cv.median():.1%}.")
P(f"- {(bp['nunique']<=3).mean():.0%} of store-products have 3 or fewer base prices; "
  f"{(bp.cv>0.10).mean():.0%} have CV above 10%.")
P("- Base price drifts upward over the three years (list-price inflation), so the ML baseline includes base_price "
  "and a linear week_index trend. Discount depth is computed against the contemporaneous base price, so drift does "
  "not leak into is_promo.")
# weekly mean base price per product (index to first week)
bpw = df.groupby(["upc", "week_index"]).base_price.mean().unstack(0)
bpw_idx = bpw / bpw.iloc[0]
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(bpw_idx.index, bpw_idx.median(axis=1), color="#1f6f8b", label="median product")
ax.fill_between(bpw_idx.index, bpw_idx.quantile(.1, axis=1), bpw_idx.quantile(.9, axis=1), alpha=.2, color="#1f6f8b", label="10-90% of products")
ax.set_xlabel("week_index"); ax.set_ylabel("base price / week-1 base price"); ax.legend(); ax.grid(alpha=.3)
ax.set_title("Base price drift over the 156 weeks"); fig.tight_layout(); fig.savefig(FIG / "phase0_base_price_drift.png", dpi=130); plt.close(fig)

# --- discount depth
P("\n## 5. Discount depth when promoted\n")
dd = df[df.price_cut].discount_depth
P(f"- Among price-cut weeks: median discount **{dd.median():.0%}**, IQR {dd.quantile(.25):.0%}-{dd.quantile(.75):.0%}, "
  f"max {dd.max():.0%}.")
P(f"- Weeks with the TPR_ONLY flag but a computed discount of 5% or less: {((df.tpr_only==1)&(df.discount_depth<=0.05)).sum():,} "
  f"(kept as promoted because the retailer flagged them).")

# --- outliers (user guide)
P("\n## 6. Outliers flagged by the user guide\n")
P(f"- units/visit > 5: {(df.units_per_visit>5).sum():,} rows; visits/household > 3: {((df.visits/df.hhs)>3).sum():,} rows; "
  f"units == 0: {(df.units==0).sum():,} rows.")
P("- Decision: keep them. Bootstrap CIs at event level absorb them; the event-study uses means and medians side by side.")

OUT.write_text("\n".join(lines), encoding="utf-8")
print("\n".join(lines))

## scripts/02_phase0_sql.py

In [ ]:
"""Phase 0, step 5: load the clean table into SQLite and do the aggregations in SQL.

Creates data/processed/promo.db with table `promo_weeks`, runs the queries in sql/phase0_queries.sql
and writes the results to outputs/phase0_sql.md.
"""
import sqlite3
import sys
from pathlib import Path

import pandas as pd

ROOT = Path(os.getcwd())
sys.path.insert(0, str(ROOT))
from src.load import build_clean, PROC  # noqa: E402

DB = PROC / "promo.db"
SQL_DIR = ROOT / "sql"
SQL_DIR.mkdir(exist_ok=True)
OUT = ROOT / "outputs" / "phase0_sql.md"

df = build_clean()
df = df.drop(columns=["promo_run_id"])  # float NaN id is not useful in SQL
df["week_end_date"] = df["week_end_date"].dt.strftime("%Y-%m-%d")
for c in ["is_promo", "price_cut", "promo_run_start", "price_imputed", "base_price_imputed"]:
    df[c] = df[c].astype(int)

if DB.exists():
    DB.unlink()
con = sqlite3.connect(DB)
df.to_sql("promo_weeks", con, index=False, chunksize=50_000)
con.execute("CREATE INDEX ix_sp ON promo_weeks(store_num, upc, week_index)")
con.execute("CREATE INDEX ix_pw ON promo_weeks(upc, week_index)")
con.commit()

QUERIES = {
"Promotion counts by store (top 10 by promo share)": """
SELECT store_num, store_segment, state,
       COUNT(*)                         AS store_product_weeks,
       SUM(is_promo)                    AS promo_weeks,
       ROUND(1.0 * SUM(is_promo) / COUNT(*), 3) AS promo_share,
       SUM(feature) AS feature_weeks, SUM(display) AS display_weeks, SUM(price_cut) AS price_cut_weeks
FROM promo_weeks
GROUP BY store_num, store_segment, state
ORDER BY promo_share DESC
LIMIT 10;
""",
"Promotion share by store segment": """
SELECT store_segment, COUNT(DISTINCT store_num) AS stores,
       ROUND(1.0 * SUM(is_promo) / COUNT(*), 3) AS promo_share,
       ROUND(AVG(CASE WHEN is_promo = 1 THEN discount_depth END), 3) AS avg_discount_when_promoted
FROM promo_weeks GROUP BY store_segment ORDER BY promo_share DESC;
""",
"Naive average lift by product (promo mean units / non-promo mean units - 1)": """
WITH p AS (
  SELECT upc, description, category,
         AVG(CASE WHEN is_promo = 1 THEN units END) AS promo_units,
         AVG(CASE WHEN is_promo = 0 THEN units END) AS base_units,
         SUM(is_promo) AS promo_weeks
  FROM promo_weeks GROUP BY upc, description, category)
SELECT upc, description, category, promo_weeks,
       ROUND(base_units, 1) AS base_units, ROUND(promo_units, 1) AS promo_units,
       ROUND(promo_units / base_units - 1, 3) AS naive_lift_pct
FROM p ORDER BY naive_lift_pct DESC;
""",
"Naive average lift by category": """
SELECT category,
       ROUND(AVG(CASE WHEN is_promo = 0 THEN units END), 1) AS base_units,
       ROUND(AVG(CASE WHEN is_promo = 1 THEN units END), 1) AS promo_units,
       ROUND(AVG(CASE WHEN is_promo = 1 THEN units END) / AVG(CASE WHEN is_promo = 0 THEN units END) - 1, 3) AS naive_lift_pct
FROM promo_weeks GROUP BY category ORDER BY naive_lift_pct DESC;
""",
"Window function: rolling 8-week pre-promo average per store-product (sample rows)": """
WITH ordered AS (
  SELECT store_num, upc, week_index, units, is_promo,
         -- units in non-promo weeks only, so the rolling mean is a clean baseline
         CASE WHEN is_promo = 0 THEN units END AS clean_units
  FROM promo_weeks),
rolled AS (
  SELECT store_num, upc, week_index, units, is_promo,
         AVG(clean_units) OVER (
             PARTITION BY store_num, upc ORDER BY week_index
             ROWS BETWEEN 8 PRECEDING AND 1 PRECEDING) AS pre8_clean_avg,
         COUNT(clean_units) OVER (
             PARTITION BY store_num, upc ORDER BY week_index
             ROWS BETWEEN 8 PRECEDING AND 1 PRECEDING) AS pre8_clean_n,
         LAG(is_promo, 1) OVER (PARTITION BY store_num, upc ORDER BY week_index) AS prev_is_promo
  FROM ordered)
SELECT store_num, upc, week_index, units, is_promo, ROUND(pre8_clean_avg, 2) AS pre8_clean_avg, pre8_clean_n,
       ROUND(units / pre8_clean_avg - 1, 3) AS lift_vs_pre8
FROM rolled
WHERE is_promo = 1 AND prev_is_promo = 0 AND pre8_clean_n >= 4
ORDER BY store_num, upc, week_index
LIMIT 12;
""",
"Window function: average lift vs rolling pre-promo baseline, by promo type (first week of each run)": """
WITH ordered AS (
  SELECT store_num, upc, week_index, units, is_promo, promo_type,
         CASE WHEN is_promo = 0 THEN units END AS clean_units
  FROM promo_weeks),
rolled AS (
  SELECT *,
         AVG(clean_units) OVER (PARTITION BY store_num, upc ORDER BY week_index
                                ROWS BETWEEN 8 PRECEDING AND 1 PRECEDING) AS pre8_clean_avg,
         COUNT(clean_units) OVER (PARTITION BY store_num, upc ORDER BY week_index
                                  ROWS BETWEEN 8 PRECEDING AND 1 PRECEDING) AS pre8_clean_n,
         LAG(is_promo, 1) OVER (PARTITION BY store_num, upc ORDER BY week_index) AS prev_is_promo
  FROM ordered)
SELECT promo_type, COUNT(*) AS events,
       ROUND(AVG(units / pre8_clean_avg - 1), 3) AS mean_lift_vs_pre8,
       ROUND(AVG(units - pre8_clean_avg), 2) AS mean_incremental_units
FROM rolled
WHERE is_promo = 1 AND prev_is_promo = 0 AND pre8_clean_n >= 4 AND pre8_clean_avg > 0
GROUP BY promo_type ORDER BY mean_lift_vs_pre8 DESC;
""",
"Window function: simultaneous promoting stores per product-week (share), ranked": """
WITH pw AS (
  SELECT upc, week_index, COUNT(*) AS n_stores, SUM(is_promo) AS n_promo
  FROM promo_weeks GROUP BY upc, week_index)
SELECT upc, week_index, n_stores, n_promo, ROUND(1.0 * n_promo / n_stores, 2) AS share_promo,
       RANK() OVER (PARTITION BY upc ORDER BY n_promo DESC) AS rank_within_product
FROM pw WHERE n_promo > 0
ORDER BY share_promo DESC, upc LIMIT 10;
""",
}

(SQL_DIR / "phase0_queries.sql").write_text(
    "\n\n".join(f"-- {k}\n{v.strip()}" for k, v in QUERIES.items()), encoding="utf-8")

lines = ["# Phase 0 SQL aggregations (data/processed/promo.db, table promo_weeks)\n",
         "Queries live in sql/phase0_queries.sql and are executed by scripts/02_phase0_sql.py.\n"]
for title, q in QUERIES.items():
    res = pd.read_sql_query(q, con)
    if "upc" in res.columns:
        res["upc"] = res["upc"].astype("int64").astype(str)
    lines.append(f"## {title}\n")
    lines.append("```sql\n" + q.strip() + "\n```\n")
    lines.append(res.to_markdown(index=False))
    lines.append("")
con.close()
OUT.write_text("\n".join(lines), encoding="utf-8")
print("\n".join(lines))